# HAC-Net, Taken Apart: a Hands-On Primer
**Series B: Deep Learning for Molecules, Post 1b** &nbsp;|&nbsp; companion *primer* notebook

HAC-Net (Kyro, Brent and Batista, *J. Chem. Inf. Model.* **63**, 1947, 2023; code at github.com/gregory-kyro/HAC-Net) predicts the binding affinity of a protein-ligand complex with one 3D convolutional network (with squeeze-and-excitation attention) and two graph convolutional networks (with attention-based aggregation).

**This primer rebuilds each building block from the paper's description, in NumPy only, runs it on a synthetic complex, and checks it against numbers the paper itself states.** It runs on a CPU in well under a minute.

> **What this notebook is not.** It does *not* train HAC-Net and does *not* reproduce its accuracy: that needs the PDBbind data and a GPU. To train and test the real model, use the authors' repository: `pip install HACNet`, and their tutorial notebook at `colab.research.google.com/github/gregory-kyro/HAC-Net/blob/main/HACNet.ipynb`. Here the weights are random and the complex is synthetic, so the *predictions* are meaningless; the *mechanics* (shapes, parameter counts, invariances, costs, attention) are what we verify.

| Section | What you do |
|---|---|
| 1 | convert affinities between pK and kcal/mol |
| 2 | build a synthetic complex; featurise, voxelise and turn it into a graph |
| 3 | 3D convolution three ways (direct, im2col, sparse "atom stamping") |
| 4 | the squeeze-and-excitation block |
| 5 | account for the 3D-CNN's parameters and compute; check the paper's "1.9 %" |
| 6 | the graph branch: attention, GRU, asymmetric aggregation; invariances; its parameters |
| 7 | the hybrid: why averaging three predictors helps |
| 8 | analyse the paper's published tables (implied label spread, memorisation, cross-validation) |
| 9 | a small *trained* demonstration: attention versus sum aggregation |

## 0. Setup

In [ ]:
import itertools, math
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from scipy.spatial import cKDTree
from scipy.optimize import minimize

plt.rcParams.update({"figure.dpi": 110, "font.size": 10, "axes.spines.top": False, "axes.spines.right": False})
COLORS = dict(blue="#0072B2", orange="#E69F00", green="#009E73", red="#D55E00", pink="#CC79A7", gray="#7f7f7f")
rng = np.random.default_rng(0)

## 1. Units: what is a pK of 1.2?
Affinity is reported as $pK=-\log_{10}K$ (with $K=K_D$ or $K_i$ in mol/L). The binding free energy is $\Delta G^\circ=RT\ln K=-RT\ln10\cdot pK$.
**Stop and predict:** the paper's RMSE on the PDBbind core set is 1.205 pK units. Is that more or less than 1 kcal/mol?

In [ ]:
R_KCAL = 0.0019872041                         # gas constant in kcal/(mol K)
KCAL_PER_PK = R_KCAL * 298.15 * math.log(10)  # kcal/mol per pK unit at 298.15 K
print(f"1 pK unit = {KCAL_PER_PK:.4f} kcal/mol = a factor of 10 in K")
for name, v in (("HAC-Net RMSE", 1.205), ("HAC-Net MAE", 0.971), ("a hypothetical 1 kcal/mol error", 1 / KCAL_PER_PK)):
    print(f"{name:34s} {v:6.3f} pK = {v * KCAL_PER_PK:5.2f} kcal/mol = a factor of {10 ** v:5.1f} in K")

## 2. A synthetic complex: features, voxels, graph
The paper's atom features (from Pafnucy) are 19 numbers per heavy atom: 9 bits for the atom type (B, C, N, O, P, S, Se, halogen, metal), hybridisation (1 to 3), the number of bonds to heavy atoms, the number of bonds to heteroatoms, 5 bits (hydrophobic, aromatic, acceptor, donor, ring), the partial charge, and $\pm1$ for protein or ligand. The graph branch adds the van der Waals radius (20 features).
We cannot download PDBbind here, so we generate a *synthetic* complex with the right shapes: a ligand (a random chain with 1.5 Å steps) inside a spherical pocket of protein atoms at a realistic density.

In [ ]:
FEATURE_NAMES = ([f"is_{t}" for t in ("B", "C", "N", "O", "P", "S", "Se", "halogen", "metal")]
                 + ["hybridisation", "bonds_heavy", "bonds_hetero", "hydrophobic", "aromatic", "acceptor", "donor", "ring", "partial_charge", "ligand_flag"])
assert len(FEATURE_NAMES) == 19

def make_complex(rng, n_prot=850, n_lig=28):
    lig = np.cumsum(rng.normal(size=(n_lig, 3)) * 1.5 / np.sqrt(3), axis=0); lig -= lig.mean(0)
    pts = []
    while len(pts) < n_prot:                                         # protein atoms: a ball of radius 14 A with a cavity around the ligand
        p = rng.uniform(-14, 14, size=3)
        if np.linalg.norm(p) > 14 or np.min(np.linalg.norm(lig - p, axis=1)) < 3.2: continue
        if pts and np.min(np.linalg.norm(np.array(pts) - p, axis=1)) < 1.2: continue
        pts.append(p)
    xyz = np.vstack([np.array(pts), lig]); is_lig = np.r_[np.zeros(n_prot), np.ones(n_lig)]
    n = len(xyz); probs = np.array([0, .62, .14, .19, .005, .03, 0, .015, 0]); probs /= probs.sum()
    typ = rng.choice(9, size=n, p=probs)
    f = np.zeros((n, 19)); f[np.arange(n), typ] = 1
    f[:, 9] = rng.integers(1, 4, n); f[:, 10] = rng.integers(1, 4, n); f[:, 11] = rng.integers(0, 3, n)
    f[:, 12:17] = rng.integers(0, 2, (n, 5)); f[:, 17] = rng.normal(0, 0.3, n); f[:, 18] = np.where(is_lig == 1, 1.0, -1.0)
    return xyz, f, is_lig, typ

xyz, feats, is_lig, typ = make_complex(rng)
VDW = {0: 1.92, 1: 1.70, 2: 1.55, 3: 1.52, 4: 1.80, 5: 1.80, 6: 1.90, 7: 1.75, 8: 2.0}      # illustrative radii by atom class
X0 = np.c_[feats, [VDW[t] for t in typ]]                                                        # 20 node features for the GCN
print(f"{len(xyz)} atoms ({int(is_lig.sum())} ligand) | 3D-CNN features per atom: {feats.shape[1]} | GCN features per atom: {X0.shape[1]}")

def voxelize(xyz, feats, L=48):
    idx = np.floor(xyz - xyz.mean(0) + L / 2).astype(int)                                     # 1 A voxels, complex centred in the grid
    inside = ((idx >= 0) & (idx < L)).all(1)
    grid = np.zeros((L, L, L, feats.shape[1]), dtype=np.float32); count = {}
    for i in np.where(inside)[0]:
        t = tuple(idx[i]); grid[t] += feats[i]; count[t] = count.get(t, 0) + 1                  # two atoms in one voxel: features are summed
    return grid, idx, inside, sum(c - 1 for c in count.values()), len(count)

grid, idx, inside, n_shared, n_occ = voxelize(xyz, feats)
print(f"voxel grid {grid.shape}: {n_occ} occupied voxels of {48 ** 3:,} ({100 * n_occ / 48 ** 3:.2f} %), {n_shared} atoms share a voxel ({100 * n_shared / len(xyz):.2f} %), {int((~inside).sum())} outside")
print("(the paper reports 0.22 % of atoms sharing a voxel and 0.01 % outside the grid for PDBbind)")

tree = cKDTree(xyz); pairs = tree.query_pairs(3.5, output_type="ndarray")                       # edges: atoms whose centres are within 3.5 A
src, dst = np.r_[pairs[:, 0], pairs[:, 1]], np.r_[pairs[:, 1], pairs[:, 0]]
deg = np.bincount(dst, minlength=len(xyz))
print(f"graph: {len(xyz)} nodes, {len(src):,} directed edges, mean degree {deg.mean():.1f} (min {deg.min()}, max {deg.max()})")
print(f"the voxel grid stores {48 ** 3 * 19:,} numbers ({48 ** 3 * 19 * 4 / 1e6:.1f} MB in float32); the graph stores {X0.size:,} node numbers and {2 * len(src):,} edge indices")

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
sel = np.abs(xyz[:, 1]) < 2.5                                                                   # a 5 A slab, for readability
ax[0].scatter(xyz[sel & (is_lig == 0), 0], xyz[sel & (is_lig == 0), 2], s=14, c=COLORS["gray"], label="protein")
ax[0].scatter(xyz[sel & (is_lig == 1), 0], xyz[sel & (is_lig == 1), 2], s=26, c=COLORS["red"], label="ligand")
ax[0].set(xlabel="x (A)", ylabel="z (A)", title="(a) A slab of the synthetic complex"); ax[0].set_aspect("equal"); ax[0].legend(fontsize=8)
ax[1].hist(deg, bins=np.arange(0, deg.max() + 2) - 0.5, color=COLORS["blue"]); ax[1].set(xlabel="neighbours within 3.5 A", ylabel="atoms", title="(b) Degree distribution of the graph")
plt.tight_layout(); plt.show()

## 3. Three ways to compute a 3D convolution
For a filter of size $k$ with $F$ input and $C$ output channels, stride $s$ and padding $p$ (Eq. 1 of the paper),
$$\text{out}[x',y',z',c]=b_c+\sum_{h,i,j=1}^{k}\sum_{f=1}^{F}\text{filter}[h,i,j,f,c]\;\text{in}[sx'+h-p,\ sy'+i-p,\ sz'+j-p,\ f],$$
and the output side length is $\lfloor(n+2p-k)/s\rfloor+1$. We implement it (a) by direct loops, (b) as one matrix product on unfolded patches (*im2col*), and (c) as a sum of **atom stamps**: because the grid is almost empty (about 1 % of voxels), the first convolution only has to add a shifted copy of the filter, weighted by the atom's features, for each atom.
**Stop and predict:** do the three agree to machine precision?

In [ ]:
def conv3d_direct(x, w, b, stride, pad):
    L = x.shape[0]; k = w.shape[0]; xp = np.pad(x, ((pad, pad),) * 3 + ((0, 0),)); n = (L + 2 * pad - k) // stride + 1
    out = np.zeros((n, n, n, w.shape[-1]))
    for i, j, l in itertools.product(range(n), repeat=3):
        patch = xp[i * stride:i * stride + k, j * stride:j * stride + k, l * stride:l * stride + k, :]
        out[i, j, l] = np.tensordot(patch, w, axes=([0, 1, 2, 3], [0, 1, 2, 3])) + b
    return out

def conv3d_im2col(x, w, b, stride, pad):
    L = x.shape[0]; k = w.shape[0]; C = w.shape[-1]; xp = np.pad(x, ((pad, pad),) * 3 + ((0, 0),)); n = (L + 2 * pad - k) // stride + 1
    win = np.lib.stride_tricks.sliding_window_view(xp, (k, k, k), axis=(0, 1, 2))[::stride, ::stride, ::stride]
    return (win.transpose(0, 1, 2, 4, 5, 6, 3).reshape(n ** 3, -1) @ w.reshape(-1, C) + b).reshape(n, n, n, C)

def conv3d_sparse(coords, vals, w, b, L, stride, pad):
    k = w.shape[0]; n = (L + 2 * pad - k) // stride + 1; out = np.zeros((n, n, n, w.shape[-1])) + b
    for p, v in zip(coords, vals):                                          # each atom "stamps" the filter around itself
        for h in itertools.product(range(k), repeat=3):
            o = np.array(p) + pad - np.array(h)
            if (o % stride == 0).all():
                o //= stride
                if ((o >= 0) & (o < n)).all(): out[tuple(o)] += v @ w[h]
    return out

xs = rng.normal(size=(8, 8, 8, 3)); w = rng.normal(size=(3, 3, 3, 3, 4)); b = rng.normal(size=4)
d1, d2 = conv3d_direct(xs, w, b, 2, 1), conv3d_im2col(xs, w, b, 2, 1)
print(f"direct vs im2col: max difference {np.abs(d1 - d2).max():.1e}  (output {d1.shape})")
sparse_pts = [(1, 2, 3), (4, 4, 4), (7, 0, 2), (3, 3, 5)]
xsp = np.zeros((8, 8, 8, 3)); xsp[tuple(zip(*sparse_pts))] = rng.normal(size=(4, 3))
a1 = conv3d_im2col(xsp, w, b, 2, 1); a2 = conv3d_sparse(sparse_pts, [xsp[p] for p in sparse_pts], w, b, 8, 2, 1)
print(f"dense vs sparse atom stamping (4 atoms): max difference {np.abs(a1 - a2).max():.1e}")

In [ ]:
out_size = lambda n, k, s, p: (n + 2 * p - k) // s + 1
print("The paper gives kernel sizes and feature-map sizes but not strides or paddings. Which (stride, padding) pairs are consistent?\n")
stages = [("SE block 1, 9x9x9", 48, 9, 24), ("residual layers, 7x7x7", 24, 7, 24), ("SE block 2, 7x7x7", 24, 7, 8), ("SE block 3, 5x5x5", 4, 5, 2)]
for name, n_in, k, n_out in stages:
    ok = [(s, p) for s in (1, 2, 3) for p in range(0, 6) if out_size(n_in, k, s, p) == n_out]
    print(f"{name:26s} {n_in:>2} -> {n_out:>2}: (stride, padding) in {ok}")

## 4. The squeeze-and-excitation (SE) block
After a convolution producing $\mathbf u$ (with $F$ channels), the block **squeezes** each channel to its spatial average, $z_c=\frac1{L^3}\sum_{ijk}u[i,j,k,c]$, passes $\mathbf z$ through two small fully connected layers, $\mathbf s=\sigma(\mathrm{ReLU}(\mathbf z W_1)W_2)$ with $W_1\in\mathbb R^{F\times F/16}$, $W_2\in\mathbb R^{F/16\times F}$, and **rescales** every channel, $x[i,j,k,c]=s_c\,u[i,j,k,c]$: a learned, input-dependent *channel-wise attention*.

In [ ]:
def se_block(u, W1, W2):
    z = u.mean(axis=(0, 1, 2))                                           # squeeze: one number per channel
    s = 1 / (1 + np.exp(-(np.maximum(z @ W1, 0) @ W2)))                  # excitation: values in (0, 1)
    return u * s, s                                                      # scale: re-weight the channels

F = 64
u = np.maximum(rng.normal(size=(6, 6, 6, F)) + rng.normal(size=F) * 0.8, 0)          # fake post-ReLU feature map with channels of different strength
W1, W2 = rng.normal(size=(F, F // 16)) * 0.3, rng.normal(size=(F // 16, F)) * 0.3
y, s = se_block(u, W1, W2)
print(f"input {u.shape} -> output {y.shape}; the {F} channel weights lie in ({s.min():.2f}, {s.max():.2f}); {F * (F // 16) * 2 + F // 16 + F:,} parameters in the two FC layers")
assert 0 < s.min() and s.max() < 1 and np.allclose(y, u * s)

# the weights depend on the INPUT: a different complex gives different channel weights
u2 = np.maximum(rng.normal(size=(6, 6, 6, F)) + rng.normal(size=F) * 0.8, 0)
print(f"channel weights for two different inputs: correlation {np.corrcoef(s, se_block(u2, W1, W2)[1])[0, 1]:.2f}")

## 5. The 3D-CNN: parameters and cost
The paper's recipe: input $48\times48\times48\times19$; SE block $9^3\times19@64$ (to $24^3\times64$); two residual layers $7^3\times64@64$; SE block $7^3\times64@128$ (to $8^3$); max-pool $2^3$ (to $4^3$); SE block $5^3\times128@256$ (to $2^3$); flatten to 2048; fully connected $2048\to100\to1$. Every convolution is followed by ReLU and batch normalisation.
The paper states that the fully connected layers hold **1.9 %** of the 3D-CNN's parameters. This lets us settle an ambiguity: does each "residual layer" contain **one** convolution or **two**?

In [ ]:
conv_p = lambda k, cin, cout: k ** 3 * cin * cout + cout       # weights + bias
bn_p = lambda c: 2 * c                                          # scale and shift
se_p = lambda F: F * (F // 16) + F // 16 + (F // 16) * F + F    # the two FC layers of an SE block

def cnn_params(convs_per_residual_layer):
    r = 2 * convs_per_residual_layer
    return {"SE block 1": conv_p(9, 19, 64) + bn_p(64) + se_p(64),
            "residual layers": r * (conv_p(7, 64, 64) + bn_p(64)),
            "SE block 2": conv_p(7, 64, 128) + bn_p(128) + se_p(128),
            "SE block 3": conv_p(5, 128, 256) + bn_p(256) + se_p(256),
            "fully connected 2048-100-1": 2048 * 100 + 100 + bn_p(100) + 100 + 1}

for r in (1, 2):
    p = cnn_params(r); tot = sum(p.values())
    print(f"{r} convolution(s) per residual layer: total {tot:,} parameters; fully connected share {100 * p['fully connected 2048-100-1'] / tot:.2f} %")
print("paper: 1.9 %  ->  each residual layer is ONE convolution with a skip connection")
params_cnn = cnn_params(1)
print()
for k, v in params_cnn.items(): print(f"   {k:28s} {v:>10,d}  {100 * v / sum(params_cnn.values()):5.1f} %")

In [ ]:
conv_macs = lambda n_out, k, cin, cout: n_out ** 3 * k ** 3 * cin * cout     # multiply-adds
macs_cnn = {"SE block 1": conv_macs(24, 9, 19, 64), "residual layers": 2 * conv_macs(24, 7, 64, 64),
            "SE block 2": conv_macs(8, 7, 64, 128), "SE block 3": conv_macs(2, 5, 128, 256), "fully connected 2048-100-1": 2048 * 100 + 100}
tot_m = sum(macs_cnn.values())
tab = pd.DataFrame({"parameters": params_cnn, "% of parameters": {k: 100 * v / sum(params_cnn.values()) for k, v in params_cnn.items()},
                    "multiply-adds": macs_cnn, "% of compute": {k: 100 * v / tot_m for k, v in macs_cnn.items()}}).round(2)
print(tab.to_string())
print(f"\ntotal compute for ONE complex: {tot_m:.3e} multiply-adds ({tot_m / 1e9:.0f} G); activations after the first layer: {24 ** 3 * 64 * 4 / 1e6:.1f} MB")
print("the parameters sit in the late, low-resolution layers; the compute sits in the early, high-resolution ones")

## 6. The graph branch
Nodes are heavy atoms with 20 features; edges join atoms within 3.5 Å. One message-passing round does three things.
1. **Message creation:** a dimension-preserving linear map $m_j=h_jW_t$.
2. **Attention aggregation** (Eq. 8): for node $i$ and the set $S_i$ made of $i$ and its neighbours, scores come from a small network $\Gamma$ (20-10-5-1, softsign after the first two layers), $\alpha_{ij}=\mathrm{softmax}_{j\in S_i}\Gamma(m_j)$, and the aggregate is $\sum_j\alpha_{ij}\,w_{ij}\,m_j$. The paper says neighbour messages are "weighted by the distance"; we take the literal reading $w_{ij}=d_{ij}$ (and $w_{ii}=1$).
3. **Update:** a GRU (Eqs. 9 to 12) combines the old features $h_i$ with the aggregate.
After four rounds, **asymmetric attentional aggregation** (Eq. 13) forms one vector for the whole complex: a gate $\mathrm{softmax}_v\,\Gamma([Y_v\|X_v])$ (40-20-128) over *all nodes*, applied to $\Omega X_v$ (20-128, from the *original* features), summed over nodes. Three fully connected layers ($128\to85\to64\to1$) give the prediction.

In [ ]:
softsign = lambda x: x / (1 + np.abs(x)); sigmoid = lambda x: 1 / (1 + np.exp(-x))

def init_gcn(rng, H=20):
    g = lambda *s: rng.normal(size=s) * 0.3
    return dict(Wmsg=g(4, H, H), G1=g(H, 10), g1=np.zeros(10), G2=g(10, 5), g2=np.zeros(5), G3=g(5, 1), g3=np.zeros(1),
                Wxr=g(H, H), Whr=g(H, H), bxr=np.zeros(H), bhr=np.zeros(H), Wxz=g(H, H), Whz=g(H, H), bxz=np.zeros(H), bhz=np.zeros(H),
                Wxn=g(H, H), Whn=g(H, H), bxn=np.zeros(H), bhn=np.zeros(H),
                A1=g(40, 20), a1=np.zeros(20), A2=g(20, 128), a2=np.zeros(128), O=g(20, 128), o=np.zeros(128),
                F1=g(128, 85), f1=np.zeros(85), F2=g(85, 64), f2=np.zeros(64), F3=g(64, 1), f3=np.zeros(1))

def seg_softmax(score, group, n_groups):                                  # softmax within groups (numerically stabilised)
    e = np.exp(score - score.max()); den = np.zeros((n_groups,) + score.shape[1:]); np.add.at(den, group, e); return e / den[group]

def gru(P, x, h):                                                          # Eqs. 9 to 12
    r = sigmoid(x @ P["Wxr"] + P["bxr"] + h @ P["Whr"] + P["bhr"]); z = sigmoid(x @ P["Wxz"] + P["bxz"] + h @ P["Whz"] + P["bhz"])
    nn = np.tanh(x @ P["Wxn"] + P["bxn"] + r * (h @ P["Whn"] + P["bhn"])); return (1 - z) * nn + z * h

def gcn_forward(P, X, src, dst, dist, weight="distance", return_attention=False):
    n = len(X); s2, d2 = np.r_[src, np.arange(n)], np.r_[dst, np.arange(n)]            # every node also belongs to its own neighbourhood
    w2 = np.r_[dist if weight == "distance" else np.ones(len(src)), np.ones(n)]; h = X.copy()
    for t in range(4):                                                                  # four message-passing rounds
        m = h @ P["Wmsg"][t]
        score = softsign(softsign(m[s2] @ P["G1"] + P["g1"]) @ P["G2"] + P["g2"]) @ P["G3"] + P["g3"]
        alpha = seg_softmax(score[:, 0], d2, n)
        agg = np.zeros_like(m); np.add.at(agg, d2, (alpha * w2)[:, None] * m[s2]); h = gru(P, agg, h)
    gate = softsign(softsign(np.c_[h, X] @ P["A1"] + P["a1"]) @ P["A2"] + P["a2"])
    wts = seg_softmax(gate, np.zeros(n, dtype=int), 1)                                  # softmax over ALL nodes, separately for each of 128 channels
    graph_vec = (wts * softsign(X @ P["O"] + P["o"])).sum(0)
    out = np.maximum(graph_vec @ P["F1"] + P["f1"], 0); out = np.maximum(out @ P["F2"] + P["f2"], 0); out = out @ P["F3"] + P["f3"]
    return (float(out[0]), alpha) if return_attention else float(out[0])

P = init_gcn(np.random.default_rng(1)); dist = np.linalg.norm(xyz[src] - xyz[dst], axis=1); n = len(xyz)
n_gcn = sum(v.size for v in P.values()); fc_gcn = P["F1"].size + P["f1"].size + P["F2"].size + P["f2"].size + P["F3"].size + P["f3"].size
print(f"GCN parameters: {n_gcn:,}; final fully connected layers {fc_gcn:,} = {100 * fc_gcn / n_gcn:.1f} % (the paper says 59.5 %; its code may contain a few parameters the text does not describe)")
pred, alpha = gcn_forward(P, X0, src, dst, dist, return_attention=True)
print(f"forward pass on the synthetic complex ({n} atoms): output {pred:.4f} (random weights: the value means nothing)")
tot = np.zeros(n); np.add.at(tot, np.r_[dst, np.arange(n)], alpha); print("attention weights of every neighbourhood sum to 1:", np.allclose(tot, 1))

In [ ]:
# invariances of the readout and of the whole branch
perm = np.random.default_rng(5).permutation(n); inv = np.argsort(perm)
pred_perm = gcn_forward(P, X0[perm], inv[src], inv[dst], dist)
print(f"relabel the atoms at random: |change in output| = {abs(pred - pred_perm):.1e}   (the graph branch cannot depend on atom order)")

R = np.linalg.qr(np.random.default_rng(9).normal(size=(3, 3)))[0]                      # a random rotation (plus possibly a reflection)
xyz_rot = xyz @ R.T; tree_r = cKDTree(xyz_rot); pairs_r = tree_r.query_pairs(3.5, output_type="ndarray")
src_r, dst_r = np.r_[pairs_r[:, 0], pairs_r[:, 1]], np.r_[pairs_r[:, 1], pairs_r[:, 0]]
pred_rot = gcn_forward(P, X0, src_r, dst_r, np.linalg.norm(xyz_rot[src_r] - xyz_rot[dst_r], axis=1))
print(f"rotate the complex: |change in output| = {abs(pred - pred_rot):.1e}   (the graph uses only distances)")

grid_rot, _, _, _, n_occ_rot = voxelize(xyz_rot, feats)
vox_a, vox_b = np.sort(np.argwhere(grid.any(-1)), axis=0), np.sort(np.argwhere(grid_rot.any(-1)), axis=0)
print(f"rotate the complex: occupied voxels {n_occ} -> {n_occ_rot}; the voxel grid itself changes (it is NOT rotation invariant): {not np.array_equal(grid, grid_rot)}")

In [ ]:
# cost of the graph branch for this complex, in multiply-adds
E2 = len(src) + n                                            # directed edges plus self loops
per_round = n * 20 * 20 + E2 * (20 * 10 + 10 * 5 + 5 * 1) + E2 * 20 + n * 6 * 20 * 20
head = n * (40 * 20 + 20 * 128) + n * 20 * 128 + 128 * 85 + 85 * 64 + 64
macs_gcn = 4 * per_round + head
print(f"GCN forward: {macs_gcn:.2e} multiply-adds for {n} atoms; 3D-CNN forward: {tot_m:.2e}; ratio {tot_m / macs_gcn:,.0f}x")

## 7. The hybrid: why average three predictors?
HAC-Net averages the outputs of the 3D-CNN and of two architecturally identical GCNs. (We read "averages the outputs" as a plain mean of three predictions, so the GCN pair carries two thirds of the weight; the paper does not give the weights explicitly.) Two identical GCNs trained from different random initialisations make partly independent errors. If each has error variance $\sigma^2$ and the correlation between their errors is $\rho$, the mean of $n$ such predictors has variance $\sigma^2[1+(n-1)\rho]/n$.

In [ ]:
sim = np.random.default_rng(11); N = 200_000
print(f"{'rho':>4} | {'one GCN':>8} | {'mean of 2 (theory)':>19} | {'mean of 2 (simulated)':>22}")
for rho in (0.0, 0.5, 0.8):
    shared = sim.normal(size=N); e = [np.sqrt(rho) * shared + np.sqrt(1 - rho) * sim.normal(size=N) for _ in range(2)]     # unit-variance errors with correlation rho
    print(f"{rho:>4} | {np.std(e[0]):>8.3f} | {np.sqrt((1 + rho) / 2):>19.3f} | {np.std((e[0] + e[1]) / 2):>22.3f}")
print("(standard deviations; with rho = 1 averaging would not help at all)")

## 8. Reading the paper's tables
These are typed in from the arXiv version of the paper (Tables 1 to 3 and Section 5). The coefficient of determination is $R^2=1-\mathrm{MSE}/\mathrm{Var}(y)$, so each row with an RMSE and an $R^2$ implies the **standard deviation of the labels in its test set**, $\sigma=\mathrm{RMSE}/\sqrt{1-R^2}$.
**Stop and predict:** three models were tested on the *same* core set. What should their implied $\sigma$ be?

In [ ]:
core = pd.DataFrame([("HAC-Net", 1.205, 0.971, 0.692, 0.846, 0.843), ("HAC-Net (trained on the v2016 refined set)", 1.259, 1.020, 0.664, 0.819, 0.814),
                     ("FAST", 1.308, 1.019, 0.638, 0.810, 0.807)], columns=["model", "RMSE", "MAE", "R2", "Pearson r", "Spearman rho"])
core["implied sigma of the labels"] = core.RMSE / np.sqrt(1 - core.R2)
core["Pearson r^2"] = core["Pearson r"] ** 2
core["RMSE / sigma"] = core.RMSE / core["implied sigma of the labels"]
print(core.round(3).to_string(index=False))
sigma_core = core["implied sigma of the labels"].mean()
print(f"\nthe three implied sigmas agree ({core['implied sigma of the labels'].min():.3f} to {core['implied sigma of the labels'].max():.3f}): the core set has a label spread of about {sigma_core:.2f} pK")
print(f"so always predicting the mean label would give RMSE ~ {sigma_core:.2f}; HAC-Net's 1.205 is {100 * 1.205 / sigma_core:.0f} % of that, and R2 <= Pearson r^2 holds in every row (the gap measures miscalibration)")

In [ ]:
# memorisation: HAC-Net trained and tested on protein-only or ligand-only data (same core set)
mem = pd.DataFrame([("protein-ligand complex", 1.205, 0.846), ("protein only", 1.742, 0.638), ("ligand only", 1.605, 0.741)], columns=["input", "RMSE", "Pearson r"])
mem["implied R2"] = 1 - (mem.RMSE / sigma_core) ** 2
mem["share of the complex's R2"] = mem["implied R2"] / mem["implied R2"].iloc[0]
print(mem.round(3).to_string(index=False))
print("\nthe ligand alone (or the protein alone) already explains much of what the complex explains: part of the performance is memorisation of")
print("which proteins and which ligands tend to bind strongly, not physics of the interaction (the concern raised by Volkov et al.)")

In [ ]:
splits = pd.DataFrame([("v2007 core (control)", 1.447, 0.598), ("protein-structure split", 1.472, 0.608), ("protein-sequence split", 1.301, 0.583),
                       ("ligand ECFP4 split", 1.240, 0.355), ("10-fold CV control (v2016 core)", 1.432, 0.567), ("10-fold CV, mean of 10 test sets", 1.473, 0.430)],
                      columns=["test set", "RMSE", "R2"])
splits["implied sigma"] = splits.RMSE / np.sqrt(1 - splits.R2); splits["RMSE / sigma"] = splits.RMSE / splits["implied sigma"]
print(splits.round(3).to_string(index=False))
print("\nthe paper notes that RMSE barely changes on the ligand and CV splits while the correlations fall. Part of the reason is visible here:")
print("those test sets have a NARROWER label spread (sigma 1.54 and 1.95 versus 2.17 and 2.18), so the same RMSE is a larger fraction of the spread.")

In [ ]:
# Table 3: ten cross-validation test sets (RMSE, MAE, R2, Pearson r, Spearman rho)
cv = np.array([[1.462, 1.152, .443, .672, .678], [1.450, 1.176, .448, .676, .672], [1.451, 1.159, .447, .669, .675], [1.497, 1.163, .414, .659, .661],
               [1.481, 1.182, .424, .654, .656], [1.454, 1.158, .444, .678, .677], [1.501, 1.187, .409, .650, .658], [1.483, 1.182, .424, .661, .680],
               [1.527, 1.219, .383, .639, .649], [1.426, 1.123, .465, .686, .697]])
mean, sd = cv.mean(0), cv.std(0, ddof=0)
print("recomputed mean:", mean.round(3), "  (paper: 1.473 1.170 0.430 0.665 0.670)")
print("recomputed std :", sd.round(3), "  (paper: 0.028 0.024 0.022 0.013 0.013)")
print(f"average coefficient of variation: {np.mean(sd / mean):.3f}   (paper: 0.026)")
sig_cv = cv[:, 0] / np.sqrt(1 - cv[:, 2])
print("implied sigma of the ten test sets:", sig_cv.round(2), "-> all about 1.95: the test sets were built to share one affinity distribution")

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3.6))
labels = ["core set\n(HAC-Net)", "v2007\ncontrol", "protein\nstructure", "protein\nsequence", "ligand\nECFP4", "10-fold CV\n(mean)"]
vals = [1.205 / sigma_core] + list(splits["RMSE / sigma"].iloc[[0, 1, 2, 3, 5]])
ax.bar(labels, vals, color=[COLORS["green"]] + [COLORS["blue"]] * 3 + [COLORS["orange"]] * 2)
ax.axhline(1.0, color="k", ls="--", lw=1); ax.text(5.45, 1.02, "predict the mean", ha="right", fontsize=8)
ax.set(ylabel="RMSE / (label standard deviation)", ylim=(0, 1.15), title="Error relative to the spread of the labels in each test set"); plt.tight_layout(); plt.show()

## 9. A small trained demonstration: attention versus sum aggregation
The paper attributes part of HAC-Net's performance to attention (Figure 5 compares it with a "vanilla" model without the attention blocks). We cannot retrain HAC-Net, but the *mechanism* is easy to isolate in a toy.
A "complex" is a set of $20$ to $60$ nodes with three features: a marker $m_v\in\{0,1\}$ (about 15 % of nodes, like atoms at the binding interface), a signal $s_v$ and a noise $c_v$. The label is the **average signal of the marked nodes only**, $y=\frac1{|M|}\sum_{v\in M}s_v+\text{noise}$.
We fit three readouts by gradient-based least squares: a **sum** $\sum_v w\cdot x_v$, a **mean** $\frac1N\sum_v w\cdot x_v$, and an **attention** readout $\sum_v\mathrm{softmax}_v(\gamma\cdot x_v)\,(\omega\cdot x_v)$ (the same form as Eq. 13, with one channel).
**Stop and predict:** which readouts can represent "average of $s$ over the marked nodes"?

In [ ]:
def make_set(rng, n_min=20, n_max=60, p=0.15):
    n = int(rng.integers(n_min, n_max)); m = (rng.random(n) < p).astype(float)
    if m.sum() == 0: m[rng.integers(n)] = 1
    s, c = rng.normal(size=n), rng.normal(size=n)
    return np.c_[m, s, c], s[m == 1].mean() + rng.normal(0, 0.1)

rng_d = np.random.default_rng(3); data = [make_set(rng_d) for _ in range(800)]; train, test = data[:400], data[400:]

def att(th, Xv):
    g, w, b = th[:3], th[3:6], th[6]; a = np.exp(Xv @ g - (Xv @ g).max()); a /= a.sum(); return (a * (Xv @ w)).sum() + b, a
readouts = {"sum": lambda th, Xv: (Xv @ th[:3]).sum() + th[3], "mean": lambda th, Xv: (Xv @ th[:3]).mean() + th[3], "attention": lambda th, Xv: att(th, Xv)[0]}
n_par = {"sum": 4, "mean": 4, "attention": 7}

def fit(fn, n_params, seed=7):
    loss = lambda th: np.mean([(fn(th, Xv) - y) ** 2 for Xv, y in train])
    return minimize(loss, np.random.default_rng(seed).normal(size=n_params) * 0.1, method="L-BFGS-B").x

def score(fn, th, dset):
    pr = np.array([fn(th, Xv) for Xv, _ in dset]); yt = np.array([y for _, y in dset])
    return np.sqrt(np.mean((pr - yt) ** 2)), 1 - np.sum((pr - yt) ** 2) / np.sum((yt - yt.mean()) ** 2)

fits = {k: fit(readouts[k], n_par[k]) for k in readouts}
print(f"baseline (predict the mean label): test RMSE {np.std([y for _, y in test]):.3f}")
res = pd.DataFrame({k: dict(zip(["train RMSE", "train R2", "test RMSE", "test R2"], [*score(readouts[k], fits[k], train), *score(readouts[k], fits[k], test)])) for k in readouts}).T
print(res.round(3).to_string())
mass = np.mean([att(fits["attention"], Xv)[1][Xv[:, 0] == 1].sum() for Xv, _ in test]); frac = np.mean([(Xv[:, 0] == 1).mean() for Xv, _ in test])
print(f"\nthe attention readout puts {100 * mass:.1f} % of its weight on the marked nodes, which are {100 * frac:.1f} % of all nodes; learned gate (marker, signal, noise): {fits['attention'][:3].round(2)}")

## 10. Your turn
**Exercise 1 (units).** What RMSE in pK corresponds to chemical accuracy (1 kcal/mol)? How does HAC-Net's 1.205 compare?

**Exercise 2 (parameters).** If the first SE block used 32 channels instead of 64 (with the rest unchanged, so the residual layers become $7^3\times32@32$ and SE block 2 becomes $7^3\times32@128$), how many parameters would the 3D-CNN have?

**Exercise 3 (the graph).** Change the distance cutoff from 3.5 Å to 5 Å: how do the number of edges and the cost of the graph branch change? Does the permutation test still pass?

**Exercise 4 (the weighting).** The paper does not say how messages are weighted by distance. Run the forward pass with `weight="none"` and with $w_{ij}=1/d_{ij}$. How large is the change in the (random-weight) output?

**Exercise 5 (the attention toy).** Raise the marker fraction from 15 % to 60 %. What happens to the gap between attention and the other readouts, and why?

**Exercise 6 (the tables).** Use Section 8 to compute the implied label spread for HAC-Net on the general-set "lower-quality" test (the paper gives only a figure for that test; what would you need to read from it?).

In [ ]:
# Starters
# Exercise 1
print("RMSE for 1 kcal/mol:", round(1 / KCAL_PER_PK, 3), "pK; HAC-Net / that =", round(1.205 * KCAL_PER_PK, 2))

# Exercise 2: first SE block with 32 channels
p32 = {"SE block 1": conv_p(9, 19, 32) + bn_p(32) + se_p(32), "residual layers": 2 * (conv_p(7, 32, 32) + bn_p(32)),
       "SE block 2": conv_p(7, 32, 128) + bn_p(128) + se_p(128), "SE block 3": conv_p(5, 128, 256) + bn_p(256) + se_p(256), "fc": 2048 * 100 + 100 + bn_p(100) + 101}
print("3D-CNN parameters with 32 channels in the first block:", f"{sum(p32.values()):,}", "vs", f"{sum(params_cnn.values()):,}")

# Exercise 3: 5 A cutoff
pairs5 = cKDTree(xyz).query_pairs(5.0, output_type="ndarray")
print("directed edges at 5 A:", 2 * len(pairs5), "vs", len(src), "at 3.5 A | mean degree", round(2 * len(pairs5) / n, 1))

# Exercise 4: other weightings
print("outputs:", {w: round(gcn_forward(P, X0, src, dst, dist, weight=w), 4) for w in ("distance", "none")},
      "| 1/d weighting:", round(gcn_forward(P, X0, src, dst, 1 / dist, weight="distance"), 4))

## 11. Self-check

In [ ]:
assert len(FEATURE_NAMES) == 19 and feats.shape[1] == 19 and X0.shape[1] == 20 and grid.shape == (48, 48, 48, 19)
assert np.abs(d1 - d2).max() < 1e-10 and np.abs(a1 - a2).max() < 1e-10
assert [(s, p) for s in (1, 2, 3) for p in range(6) if out_size(48, 9, s, p) == 24] == [(2, 4)] and (1, 3) in [(s, p) for s in (1, 2, 3) for p in range(6) if out_size(24, 7, s, p) == 24]
assert sum(cnn_params(1).values()) == 10_820_333 and abs(100 * cnn_params(1)["fully connected 2048-100-1"] / 10_820_333 - 1.9) < 0.01
assert abs(100 * cnn_params(2)["fully connected 2048-100-1"] / sum(cnn_params(2).values()) - 1.9) > 0.3
assert n_gcn == 27_121 and np.allclose(tot, 1) and abs(pred - pred_perm) < 1e-9 and abs(pred - pred_rot) < 1e-9
assert abs(KCAL_PER_PK - 1.3642) < 1e-3 and abs(core["implied sigma of the labels"].max() - core["implied sigma of the labels"].min()) < 0.01
assert abs(mem["share of the complex's R2"].iloc[1] - 0.51) < 0.01 and abs(mem["share of the complex's R2"].iloc[2] - 0.66) < 0.01
assert np.allclose(mean, [1.473, 1.170, 0.430, 0.664, 0.670], atol=1.5e-3) and abs(np.mean(sd / mean) - 0.026) < 0.002
assert res.loc["attention", "test RMSE"] < 0.15 and res.loc["sum", "test RMSE"] > 0.3 and mass > 0.9
print("All checks passed: you reproduced the numbers in the post.")